# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mujahid1hm/flyrank-ai-/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

This lane is a page-level content contract. One row represents one content item (a pseudonymized page) observed over its trailing 90-day traffic history, with the model making a page-level decision about whether to refresh or prioritize it. The training window is a mid-panel month such as `2026-03`; the final month is kept sealed as a test window, and the label is built from later observed trend direction rather than from product flags.


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from pathlib import Path
import pandas as pd

search_roots = [Path.cwd(), *Path.cwd().parents]

def find_dataset():
    for root in search_roots:
        candidate = root / "data" / "raw" / "content_refresh_anonymized.csv"
        if candidate.exists():
            return candidate
    for root in search_roots:
        matches = list(root.rglob("content_refresh_anonymized.csv"))
        if matches:
            return matches[0]
    return None

candidate = find_dataset()
if candidate is None:
    raise FileNotFoundError("Starter dataset not found under data/raw/")

df = pd.read_csv(candidate)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"Rows: {len(df):,}")
print(f"Unique content_id values: {df['content_id'].nunique():,}")
print(f"Duplicate content_id rows: {df.duplicated(subset=['content_id']).sum():,}")
print("One row = one content item / page in the trailing-90-day snapshot.")
print(f"Declining label rate: {df['is_declining_label'].mean():.3f} ({df['is_declining_label'].mean() * 100:.1f}%)")


Rows: 30,000
Unique content_id values: 30,000
Duplicate content_id rows: 0
One row = one content item / page in the trailing-90-day snapshot.
Declining label rate: 0.542 (54.2%)


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature fields (known before the decision): `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `word_count`, `content_age_days`, `search_volume`, `competition`, `cpc`.

Label / proxy: `is_declining_label`, derived from the observed `trend_direction` value; it is the outcome we predict and never a feature.

Context / grouping fields: `content_id`, `client_id`, `content_type`, `main_intent`, `provider_used`, `model_used`.

Excluded fields: `trend_direction` and `trend_pct` are excluded because they are part of the label construction; any future-looking engagement or conversion metrics are excluded to avoid leakage; product-decision flags are excluded because they sit on the other side of the decision we are trying to predict.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from pathlib import Path
import pandas as pd

search_roots = [Path.cwd(), *Path.cwd().parents]

def find_dataset():
    for root in search_roots:
        candidate = root / "data" / "raw" / "content_refresh_anonymized.csv"
        if candidate.exists():
            return candidate
    for root in search_roots:
        matches = list(root.rglob("content_refresh_anonymized.csv"))
        if matches:
            return matches[0]
    return None

candidate = find_dataset()
if candidate is None:
    raise FileNotFoundError("Starter dataset not found under data/raw/")

df = pd.read_csv(candidate)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

feature_fields = [
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "word_count",
    "content_age_days",
    "search_volume",
    "competition",
    "cpc",
]
label_fields = ["is_declining_label", "trend_direction", "trend_pct"]
context_fields = ["content_id", "client_id", "content_type", "main_intent", "provider_used", "model_used"]
excluded_fields = ["trend_direction", "trend_pct", "is_declining_label"]

print("FEATURES")
print(feature_fields)
print("\nLABEL / PROXY")
print(label_fields)
print("\nCONTEXT")
print(context_fields)
print("\nEXCLUDED")
print(excluded_fields)
print("\nReason for exclusion: label-derived and future/decision-side fields would leak target information or distort inference.")


FEATURES
['ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'word_count', 'content_age_days', 'search_volume', 'competition', 'cpc']

LABEL / PROXY
['is_declining_label', 'trend_direction', 'trend_pct']

CONTEXT
['content_id', 'client_id', 'content_type', 'main_intent', 'provider_used', 'model_used']

EXCLUDED
['trend_direction', 'trend_pct', 'is_declining_label']

Reason for exclusion: label-derived and future/decision-side fields would leak target information or distort inference.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The checks below confirm the grain, the row volume, and the availability of valid records. The same logic applies to the warehouse table when the month-level dataset is available: the grain must be unique per content item, the row count and date span must match the panel window, and the valid/surviving rows should be counted only after applying the `IS TRUE` availability filter.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
from pathlib import Path
import pandas as pd

search_roots = [Path.cwd(), *Path.cwd().parents]

def find_dataset():
    for root in search_roots:
        candidate = root / "data" / "raw" / "content_refresh_anonymized.csv"
        if candidate.exists():
            return candidate
    for root in search_roots:
        matches = list(root.rglob("content_refresh_anonymized.csv"))
        if matches:
            return matches[0]
    return None

candidate = find_dataset()
if candidate is None:
    raise FileNotFoundError("Starter dataset not found under data/raw/")

df = pd.read_csv(candidate)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print("Fact 1 — Grain check")
content_dups = df.groupby("content_id").size().reset_index(name="row_count")
content_dups = content_dups[content_dups["row_count"] > 1]
print(content_dups.head())
print(f"Duplicate content_id groups: {len(content_dups)}")
print("Interpretation: if the grain is one row per page, this should be empty.")

print("\nFact 2 — Volume & span")
print({
    "total_rows": len(df),
    "unique_content_items": df["content_id"].nunique(),
    "starter_csv_only": True,
})
print("Warehouse version: in the real HF panel, use MIN/MAX(report_date) over the 2026-03 slice; the local starter file does not carry report_date.")

print("\nFact 3 — Availability / valid rows")
valid_flag = df["avg_position"].notna() & df["ctr"].notna() & df["engagement_rate"].notna()
print(f"Rows with valid position, CTR, and engagement: {valid_flag.sum():,}")
print(f"Rows dropped by filter: {(~valid_flag).sum():,}")
print("Interpretation: only rows with a valid observed signal should survive the availability check.")


Fact 1 — Grain check
Empty DataFrame
Columns: [content_id, row_count]
Index: []
Duplicate content_id groups: 0
Interpretation: if the grain is one row per page, this should be empty.

Fact 2 — Volume & span
{'total_rows': 30000, 'unique_content_items': 30000, 'starter_csv_only': True}
Warehouse version: in the real HF panel, use MIN/MAX(report_date) over the 2026-03 slice; the local starter file does not carry report_date.

Fact 3 — Availability / valid rows
Rows with valid position, CTR, and engagement: 30,000
Rows dropped by filter: 0
Interpretation: only rows with a valid observed signal should survive the availability check.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Data Slice Limitation: This dataset slice only captures standard web traffic. App-based traffic and bot-filtered impressions are excluded, which may understate total impression volume for mobile-first or app-heavy pages. The model is therefore directional and decision-support only, not a claim about total demand in every channel.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression

search_roots = [Path.cwd(), *Path.cwd().parents]

def find_dataset():
    for root in search_roots:
        candidate = root / "data" / "raw" / "content_refresh_anonymized.csv"
        if candidate.exists():
            return candidate
    for root in search_roots:
        matches = list(root.rglob("content_refresh_anonymized.csv"))
        if matches:
            return matches[0]
    return None

candidate = find_dataset()
if candidate is None:
    raise FileNotFoundError("Starter dataset not found under data/raw/")

df = pd.read_csv(candidate)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

sample = df.sample(n=min(5000, len(df)), random_state=42).copy()

sample["f1_recent_ctr"] = sample["ctr"].fillna(0)
sample["f2_content_age_days"] = sample["content_age_days"].fillna(0)
sample["f3_avg_position"] = sample["avg_position"].fillna(0)
sample["f4_word_count"] = sample["word_count"].fillna(0)
sample["f5_engagement_rate"] = sample["engagement_rate"].fillna(0)

honest_features = [
    "f1_recent_ctr",
    "f2_content_age_days",
    "f3_avg_position",
    "f4_word_count",
    "f5_engagement_rate",
]

X_honest = sample[honest_features].fillna(0)
y = sample["is_declining_label"].astype(int)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_honest, y)
honest_accuracy = clf.score(X_honest, y)
print(f"Honest feature model accuracy: {honest_accuracy:.4f}")

# Intentional leakage trap: duplicate the label as a feature.
sample["LEAKED_feature_decline_label"] = sample["is_declining_label"].astype(float)
X_leaked = sample[honest_features + ["LEAKED_feature_decline_label"]].fillna(0)
clf.fit(X_leaked, y)
leaked_accuracy = clf.score(X_leaked, y)
print(f"Leaked feature model accuracy (TRAP!): {leaked_accuracy:.4f}")

sample.drop(columns=["LEAKED_feature_decline_label"], inplace=True)
print("Leaked feature removed; model is back to the honest baseline.")

print("\nFive honest features and why they are knowable at decision time:")
for name, reason in {
    "f1_recent_ctr": "Knowable at decision moment because it is computed from prior observed page traffic and CTR in the trailing window.",
    "f2_content_age_days": "Knowable at decision moment because content age is stored in the page metadata and is fixed before decision time.",
    "f3_avg_position": "Knowable at decision moment because average position is measured from historical Search Console performance before the refresh decision.",
    "f4_word_count": "Knowable at decision moment because the editorial content length is available from the published page metadata.",
    "f5_engagement_rate": "Knowable at decision moment because it is derived from prior engagement behavior observed before the decision is made.",
}.items():
    print(f"- {name}: {reason}")


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.